# Preprocessing Pipeline: Tinggi Muka Air (TMA)

Notebook preprocessing untuk pembersihan target (`tma_mdpl`), resampling data lingkungan (`1H -> 6H`), feature engineering (lag, delta, rolling hujan, waktu siklikal), normalisasi per-pos & global, encoding kategorikal, serta pembentukan *windowed sequences*.

**Alur Pipeline:**
1. **Konfigurasi & Setup** — Mengatur direktori dataset (`datasets/`) dan output (`artifacts/preprocessing/`).
2. **Load Data** — Memuat `train.csv`, `test.csv`, dan `data_pendukung/data_lingkungan.csv`.
3. **Pembersihan Target (`tma_mdpl`)** — Menangani sensor drop ke 0 dan spike ekstrem berbasis IQR per pos.
4. **Resampling Lingkungan (`1H -> 6H`)** — Agregasi kausal (right-closed) selaras dengan jadwal pengamatan TMA (06:00, 12:00, 18:00).
5. **Penggabungan Target & Lingkungan** — Join data train dan test dengan fitur lingkungan 6-jam.
6. **Feature Engineering** — Fitur waktu siklikal (`sin`/`cos`), lag TMA, delta TMA, dan akumulasi rolling hujan.
7. **Normalisasi Target per Pos** — `StandardScaler` di-fit khusus pada data train masing-masing pos (mencegah data leakage).
8. **Normalisasi Fitur Numerik Global** — `StandardScaler` di-fit pada fitur numerik train.
9. **Label Encoding Kategorikal** — Encoding `nama_pos` dan `landcover_name`.
10. **Windowed Sequences & Simpan ke Disk** — Membentuk tensor sekuensial `(N, window_size, n_features)` dan menyimpan artefak.

## 0. Import & Konfigurasi Path


In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle
import warnings
warnings.filterwarnings('ignore')

# ────────────────────────────────────────────────
# KONFIGURASI
# ────────────────────────────────────────────────
BASE_DIR     = os.path.abspath('.')
DATA_DIR     = os.path.join(BASE_DIR, 'datasets')
ENV_PATH     = os.path.join(DATA_DIR, 'data_pendukung', 'data_lingkungan.csv')
TRAIN_PATH   = os.path.join(DATA_DIR, 'train.csv')
TEST_PATH    = os.path.join(DATA_DIR, 'test.csv')
OUT_DIR      = os.path.join(BASE_DIR, 'artifacts', 'preprocessing')
os.makedirs(OUT_DIR, exist_ok=True)

WINDOW_SIZE  = 8    # 8 langkah × 6 jam = 48 jam historis per sampel
TARGET_COL   = 'tma_mdpl'

# Fitur numerik dari env (setelah resample 6H)
ENV_NUMERICAL = [
    'rainfall_mm_sum',          # akumulasi hujan 6 jam
    'rainfall_openmeteo_mm_sum',
    'rainfall_max_24h_mm_max',
    'solar_radiation_mj_m2_sum',
    'temperature_c_mean',
    'humidity_pct_mean',
    'wind_speed_kmh_mean',
    'dew_point_c_mean',
    'cloud_cover_pct_mean',
    'soil_moisture_0_7cm_mean',
    'soil_moisture_7_28cm_mean',
    'soil_moisture_28_100cm_mean',
    'soil_moisture_100_255cm_mean',
    'surface_pressure_hpa_mean',
    'pressure_msl_hpa_mean',
    # Fitur makro-iklim (slow-varying, tapi kita sertakan sebagai nilai konteks)
    'rmm1_mean',
    'rmm2_mean',
    'mjo_amplitude_mean',
    'nino_34_mean',
]


## 1. Pemuatan Data (Train, Test, dan Lingkungan)


In [ ]:
# ────────────────────────────────────────────────
# STEP 1: LOAD DATA
# ────────────────────────────────────────────────
print("=" * 60)
print("STEP 1: Loading data...")
print("=" * 60)

df_train_raw = pd.read_csv(TRAIN_PATH)
df_train_raw['datetime'] = pd.to_datetime(df_train_raw['datetime'])
df_train_raw = df_train_raw.sort_values(['nama_pos', 'datetime']).reset_index(drop=True)

df_test_raw = pd.read_csv(TEST_PATH)
df_test_raw['datetime'] = pd.to_datetime(df_test_raw['id'].str[:19])
df_test_raw['nama_pos'] = df_test_raw['id'].str[22:]
df_test_raw = df_test_raw.sort_values(['nama_pos', 'datetime']).reset_index(drop=True)

df_env_raw = pd.read_csv(ENV_PATH)
df_env_raw['datetime'] = pd.to_datetime(df_env_raw['datetime'])
df_env_raw = df_env_raw.sort_values(['nama_pos', 'datetime']).reset_index(drop=True)

print(f"Train rows    : {len(df_train_raw):,}  |  Pos: {df_train_raw['nama_pos'].nunique()}")
print(f"Test rows     : {len(df_test_raw):,}  |  Pos: {df_test_raw['nama_pos'].nunique()}")
print(f"Env rows      : {len(df_env_raw):,}  |  Pos: {df_env_raw['nama_pos'].nunique()}")


## 2. Pembersihan Target (`tma_mdpl`) per Pos


In [ ]:
# ────────────────────────────────────────────────
# STEP 2: BERSIHKAN TARGET tma_mdpl
# ────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2: Cleaning target tma_mdpl...")
print("=" * 60)

def clean_tma(df, iqr_mult=4.0, drop_thresh=0.01):
    """
    Bersihkan TMA per pos:
    - Spike ekstrem (sensor glitch): deteksi IQR -> set NaN -> interpolasi linear
    - Drop ke 0/sangat kecil (sensor offline): set NaN -> interpolasi linear
    
    Strategi BERBEDA sesuai jenis anomali (sesuai analisis EDA):
    - Spike: cukup interpolasi singkat (durasi 1 step)
    - Drop ke 0: NaN -> interpolasi (sensor mati, bukan noise fisik)
    """
    cleaned = []
    stats = []
    
    for pos, grp in df.groupby('nama_pos', sort=False):
        grp = grp.copy().sort_values('datetime').reset_index(drop=True)
        series = grp[TARGET_COL].copy()
        
        # [a] Sensor drop ke 0 atau negatif -> NaN
        mask_drop = series <= drop_thresh
        n_drop = mask_drop.sum()
        series[mask_drop] = np.nan
        
        # [b] Spike IQR
        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)
        iqr = q3 - q1
        upper = q3 + iqr_mult * iqr
        # Lower hanya untuk kasus yang sangat jauh di bawah (bukan sensor drop yg sudah ditangani)
        lower = q1 - iqr_mult * iqr
        mask_spike = (series > upper) | (series < lower)
        n_spike = mask_spike.sum()
        series[mask_spike] = np.nan
        
        # Interpolasi linear, lalu forward/backward fill untuk ujung-ujung
        series = series.interpolate(method='linear', limit_direction='both')
        series = series.ffill().bfill()
        
        grp[TARGET_COL] = series
        cleaned.append(grp)
        stats.append({
            'nama_pos': pos,
            'n_total': len(grp),
            'n_drop_to_zero': int(n_drop),
            'n_spike': int(n_spike),
            'pct_cleaned': round(100 * (n_drop + n_spike) / len(grp), 2)
        })
    
    df_clean = pd.concat(cleaned, ignore_index=True)
    df_stats = pd.DataFrame(stats)
    return df_clean, df_stats

df_train_clean, clean_stats = clean_tma(df_train_raw)

print("\nHasil pembersihan per pos:")
print(clean_stats[clean_stats['pct_cleaned'] > 0].to_string(index=False))
print(f"\nTotal poin yang dibersihkan: {(clean_stats['n_drop_to_zero'] + clean_stats['n_spike']).sum()}")
clean_stats.to_csv(os.path.join(OUT_DIR, 'cleaning_stats.csv'), index=False)


## 3. Resampling Data Lingkungan (`1H -> 6H`)


In [ ]:
# ────────────────────────────────────────────────
# STEP 3: RESAMPLE ENV 1H -> 6H
# ────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 3: Resampling env 1H -> 6H...")
print("=" * 60)

# Strategi agregasi per kolom
AGG_RULES = {
    'rainfall_mm': 'sum',
    'rainfall_openmeteo_mm': 'sum',
    'rainfall_max_24h_mm': 'max',
    'solar_radiation_mj_m2': 'sum',
    'temperature_c': 'mean',
    'humidity_pct': 'mean',
    'wind_speed_kmh': 'mean',
    'wind_direction_deg': 'mean',
    'dew_point_c': 'mean',
    'cloud_cover_pct': 'mean',
    'soil_moisture_0_7cm': 'mean',
    'soil_moisture_7_28cm': 'mean',
    'soil_moisture_28_100cm': 'mean',
    'soil_moisture_100_255cm': 'mean',
    'surface_pressure_hpa': 'mean',
    'pressure_msl_hpa': 'mean',
    'rmm1': 'mean',
    'rmm2': 'mean',
    'mjo_amplitude': 'mean',
    'mjo_phase': 'first',   # kategorikal, ambil nilai pertama
    'mjo_active': 'first',
    'nino_34': 'mean',
    'built_surface_m2': 'first',  # statis
    'landcover_class': 'first',   # statis
    'landcover_name': 'first',    # statis
}

env_resampled_parts = []
for pos, grp in df_env_raw.groupby('nama_pos', sort=False):
    grp = grp.set_index('datetime').sort_index()
    # label='right': window [00:00–06:00) -> index 06:00 (sesuai timestamp target)
    grp_6h = grp.resample('6h', label='right').agg(AGG_RULES)
    grp_6h.columns = [f"{c}_{agg}" for c, agg in AGG_RULES.items()]
    grp_6h['nama_pos'] = pos
    grp_6h = grp_6h.reset_index()
    env_resampled_parts.append(grp_6h)

df_env_6h = pd.concat(env_resampled_parts, ignore_index=True)

# Rename kolom agar konsisten
df_env_6h = df_env_6h.rename(columns={
    'mjo_phase_first': 'mjo_phase',
    'mjo_active_first': 'mjo_active',
    'landcover_class_first': 'landcover_class',
    'landcover_name_first': 'landcover_name',
    'built_surface_m2_first': 'built_surface_m2',
})

print(f"Env 6H shape: {df_env_6h.shape}")
print(f"Env 6H columns: {list(df_env_6h.columns)}")


## 4. Penggabungan Data Target dan Lingkungan


In [ ]:
# ────────────────────────────────────────────────
# STEP 4: GABUNG TARGET + ENV
# ────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 4: Merging target with env features...")
print("=" * 60)

# Tambahkan is_test flag
df_train_clean['is_test'] = 0
df_train_clean['split'] = 'train'
df_test_raw['is_test'] = 1
df_test_raw['split'] = 'test'
df_test_raw[TARGET_COL] = np.nan  # Target belum diketahui untuk test

# Gabung train + test menjadi satu dataframe besar
df_all = pd.concat([
    df_train_clean[['datetime', 'nama_pos', TARGET_COL, 'is_test', 'split']],
    df_test_raw[['datetime', 'nama_pos', TARGET_COL, 'is_test', 'split']]
], ignore_index=True)
df_all = df_all.sort_values(['nama_pos', 'datetime']).reset_index(drop=True)

# Merge dengan env
df_merged = df_all.merge(
    df_env_6h,
    on=['datetime', 'nama_pos'],
    how='left'
)

n_missing_env = df_merged[ENV_NUMERICAL[0]].isnull().sum()
print(f"Rows after merge: {len(df_merged):,}")
print(f"Missing env rows (datetime/pos tidak ada di env): {n_missing_env}")


## 5. Feature Engineering (Waktu Siklikal, Lag, dan Rolling)


In [ ]:
# ────────────────────────────────────────────────
# STEP 5: FEATURE ENGINEERING
# ────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 5: Feature Engineering...")
print("=" * 60)

def add_time_features(df):
    """Fitur siklikal waktu."""
    df = df.copy()
    df['hour'] = df['datetime'].dt.hour
    df['month'] = df['datetime'].dt.month
    df['dayofyear'] = df['datetime'].dt.dayofyear
    # Sin/cos encoding
    df['hour_sin']      = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos']      = np.cos(2 * np.pi * df['hour'] / 24)
    df['month_sin']     = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos']     = np.cos(2 * np.pi * df['month'] / 12)
    df['dayofyear_sin'] = np.sin(2 * np.pi * df['dayofyear'] / 365)
    df['dayofyear_cos'] = np.cos(2 * np.pi * df['dayofyear'] / 365)
    return df

def add_lag_rolling_features(df, lags=[1, 2, 3, 4, 8], rain_windows=[4, 8, 16]):
    """
    Lag TMA dan rolling curah hujan per pos.
    Semua shift dilakukan secara backward (no leakage).
    """
    df = df.copy().sort_values(['nama_pos', 'datetime'])
    
    # Lag TMA (target saat t-k)
    for lag in lags:
        df[f'tma_lag_{lag}'] = df.groupby('nama_pos')[TARGET_COL].shift(lag)
    
    # Delta TMA (perubahan dari lag ke lag sebelumnya)
    df['tma_delta_1'] = df.groupby('nama_pos')[TARGET_COL].shift(1).sub(
        df.groupby('nama_pos')[TARGET_COL].shift(2)
    )
    df['tma_delta_2'] = df.groupby('nama_pos')[TARGET_COL].shift(2).sub(
        df.groupby('nama_pos')[TARGET_COL].shift(3)
    )
    
    # Rolling curah hujan (akumulasi ke belakang)
    for w in rain_windows:
        df[f'rain_rolling_sum_{w}'] = (
            df.groupby('nama_pos')['rainfall_mm_sum']
            .transform(lambda x: x.shift(1).rolling(w, min_periods=1).sum())
        )
        df[f'rain_openmeteo_rolling_sum_{w}'] = (
            df.groupby('nama_pos')['rainfall_openmeteo_mm_sum']
            .transform(lambda x: x.shift(1).rolling(w, min_periods=1).sum())
        )
    
    # Rolling TMA std (indikasi volatilitas/variabilitas)
    df['tma_rolling_std_8'] = (
        df.groupby('nama_pos')[TARGET_COL]
        .transform(lambda x: x.shift(1).rolling(8, min_periods=2).std())
    )
    
    return df

df_merged = add_time_features(df_merged)
df_merged = add_lag_rolling_features(df_merged)

TIME_FEATURES = [
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos',
    'dayofyear_sin', 'dayofyear_cos'
]
LAG_FEATURES = (
    [f'tma_lag_{l}' for l in [1, 2, 3, 4, 8]] +
    ['tma_delta_1', 'tma_delta_2', 'tma_rolling_std_8'] +
    [f'rain_rolling_sum_{w}' for w in [4, 8, 16]] +
    [f'rain_openmeteo_rolling_sum_{w}' for w in [4, 8, 16]]
)
STATIC_NUMERICAL = ['built_surface_m2']
ALL_NUMERICAL = ENV_NUMERICAL + LAG_FEATURES + TIME_FEATURES + STATIC_NUMERICAL

print(f"Total fitur numerik: {len(ALL_NUMERICAL)}")
print(f"  - Env features   : {len(ENV_NUMERICAL)}")
print(f"  - Lag/Rolling    : {len(LAG_FEATURES)}")
print(f"  - Time cyclical  : {len(TIME_FEATURES)}")
print(f"  - Static numerical: {len(STATIC_NUMERICAL)}")


## 6. Normalisasi Target (`tma_mdpl`) per Pos


In [ ]:
# ────────────────────────────────────────────────
# STEP 6: NORMALISASI PER-POS (TARGET)
# ────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 6: Per-pos normalization for target tma_mdpl...")
print("=" * 60)

# CRITICAL: Fit scaler HANYA pada data train (jangan pakai test!)
scalers_tma = {}
df_merged['tma_scaled'] = np.nan

for pos in df_merged['nama_pos'].unique():
    mask_train_pos = (df_merged['nama_pos'] == pos) & (df_merged['split'] == 'train')
    mask_all_pos   = (df_merged['nama_pos'] == pos)
    
    train_vals = df_merged.loc[mask_train_pos, TARGET_COL].values.reshape(-1, 1)
    
    if len(train_vals) == 0:
        continue
    
    scaler = StandardScaler()
    scaler.fit(train_vals)
    scalers_tma[pos] = scaler
    
    # Transform semua (train + test, tapi test masih NaN di tma_mdpl, jadi tidak masalah)
    all_vals = df_merged.loc[mask_all_pos, TARGET_COL].values.reshape(-1, 1)
    df_merged.loc[mask_all_pos, 'tma_scaled'] = scaler.transform(all_vals).ravel()

# Simpan scalers
with open(os.path.join(OUT_DIR, 'scalers_tma_per_pos.pkl'), 'wb') as f:
    pickle.dump(scalers_tma, f)

print(f"Scalers disimpan untuk {len(scalers_tma)} pos")
print("Contoh mean/std:")
for pos, sc in list(scalers_tma.items())[:3]:
    print(f"  {pos}: mean={sc.mean_[0]:.3f}, std={sc.scale_[0]:.3f}")


## 7. Normalisasi Fitur Numerik Global


In [ ]:
# ────────────────────────────────────────────────
# STEP 7: NORMALISASI FITUR NUMERIK (GLOBAL)
# ────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 7: Global normalization for numerical features...")
print("=" * 60)

# Fit HANYA pada train
mask_train = df_merged['split'] == 'train'
scaler_num = StandardScaler()

# Isi NaN sementara dengan 0 agar bisa fit (NaN di lag/rolling wajar di baris pertama per pos)
train_num_data = df_merged.loc[mask_train, ALL_NUMERICAL].fillna(0)
scaler_num.fit(train_num_data)

# Transform semua data
df_merged[ALL_NUMERICAL] = scaler_num.transform(df_merged[ALL_NUMERICAL].fillna(0))

with open(os.path.join(OUT_DIR, 'scaler_numerical_global.pkl'), 'wb') as f:
    pickle.dump(scaler_num, f)

print(f"Global scaler disimpan untuk {len(ALL_NUMERICAL)} fitur")


## 8. Label Encoding Fitur Kategorikal


In [ ]:
# ────────────────────────────────────────────────
# STEP 8: LABEL ENCODING KATEGORIK
# ────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 8: Label encoding categorical features...")
print("=" * 60)

le_pos = LabelEncoder()
le_lc = LabelEncoder()

df_merged['pos_id']       = le_pos.fit_transform(df_merged['nama_pos'])
df_merged['landcover_id'] = le_lc.fit_transform(df_merged['landcover_name'].fillna('Unknown'))

with open(os.path.join(OUT_DIR, 'label_encoder_pos.pkl'), 'wb') as f:
    pickle.dump(le_pos, f)
with open(os.path.join(OUT_DIR, 'label_encoder_landcover.pkl'), 'wb') as f:
    pickle.dump(le_lc, f)

print(f"Jumlah ID pos      : {df_merged['pos_id'].nunique()}  -> {list(le_pos.classes_)[:5]} ...")
print(f"Jumlah landcover   : {df_merged['landcover_id'].nunique()} -> {list(le_lc.classes_)}")

# Simpan mapping untuk referensi
pos_mapping = pd.DataFrame({'nama_pos': le_pos.classes_, 'pos_id': range(len(le_pos.classes_))})
pos_mapping.to_csv(os.path.join(OUT_DIR, 'pos_id_mapping.csv'), index=False)
lc_mapping = pd.DataFrame({'landcover_name': le_lc.classes_, 'landcover_id': range(len(le_lc.classes_))})
lc_mapping.to_csv(os.path.join(OUT_DIR, 'landcover_id_mapping.csv'), index=False)


## 9. Pembentukan Windowed Sequences


In [ ]:
# ────────────────────────────────────────────────
# STEP 9: BUAT WINDOWED SEQUENCES
# ────────────────────────────────────────────────
print("\n" + "=" * 60)
print(f"STEP 9: Building windowed sequences (window={WINDOW_SIZE})...")
print("=" * 60)

FEATURE_COLS_SEQ  = ALL_NUMERICAL   # Fitur untuk sequence (LSTM input)
CATEGORICAL_COLS  = ['pos_id', 'landcover_id']  # Embedding lookup
TARGET_SCALED_COL = 'tma_scaled'

def build_sequences(df, window_size, feature_cols, cat_cols, target_col, is_test=False):
    """
    Buat sliding window per pos.
    Untuk setiap t, window historis = [t-window_size, ..., t-1]
    Target = nilai di t.
    
    Returns:
      X_seq    : (N, window_size, n_features)  - input sekuens untuk LSTM
      X_cat    : (N, n_cat)                    - ID kategorik untuk embedding
      y        : (N,)                          - target tma_scaled
      meta     : DataFrame dengan datetime, nama_pos, split
    """
    X_seq_list, X_cat_list, y_list, meta_list = [], [], [], []
    
    for pos, grp in df.groupby('nama_pos', sort=False):
        grp = grp.sort_values('datetime').reset_index(drop=True)
        n = len(grp)
        
        if n <= window_size:
            print(f"  SKIP {pos}: hanya {n} baris, tidak cukup untuk window {window_size}")
            continue
        
        features  = grp[feature_cols].values.astype(np.float32)
        cat_vals  = grp[cat_cols].values[window_size:]        # ambil nilai saat t
        targets   = grp[target_col].values[window_size:]
        datetimes = grp['datetime'].values[window_size:]
        splits    = grp['split'].values[window_size:]
        
        for i in range(n - window_size):
            window = features[i : i + window_size]            # historis
            X_seq_list.append(window)
            X_cat_list.append(cat_vals[i])
            y_list.append(targets[i])
            meta_list.append({
                'datetime': datetimes[i],
                'nama_pos': pos,
                'split': splits[i]
            })
    
    X_seq = np.array(X_seq_list, dtype=np.float32)
    X_cat = np.array(X_cat_list, dtype=np.int32)
    y     = np.array(y_list, dtype=np.float32)
    meta  = pd.DataFrame(meta_list)
    
    return X_seq, X_cat, y, meta

X_seq, X_cat, y, meta = build_sequences(
    df_merged, WINDOW_SIZE, FEATURE_COLS_SEQ, CATEGORICAL_COLS, TARGET_SCALED_COL
)

# Split train/test kembali
is_train_mask = meta['split'] == 'train'
is_test_mask  = meta['split'] == 'test'

X_seq_train = X_seq[is_train_mask]
X_cat_train = X_cat[is_train_mask]
y_train      = y[is_train_mask]
meta_train   = meta[is_train_mask].reset_index(drop=True)

X_seq_test = X_seq[is_test_mask]
X_cat_test = X_cat[is_test_mask]
y_test      = y[is_test_mask]   # NaN (target tidak diketahui)
meta_test   = meta[is_test_mask].reset_index(drop=True)

print(f"\nHasil windowing:")
print(f"  X_seq_train  : {X_seq_train.shape}  (N_samples, window, n_features)")
print(f"  X_cat_train  : {X_cat_train.shape}  (N_samples, n_cat_embed)")
print(f"  y_train      : {y_train.shape}")
print(f"  X_seq_test   : {X_seq_test.shape}")
print(f"  X_cat_test   : {X_cat_test.shape}")


## 10. Penyimpanan Artefak Preprocessing ke Disk


In [ ]:
# ────────────────────────────────────────────────
# STEP 10: SIMPAN HASIL KE DISK
# ────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 10: Saving preprocessed data to disk...")
print("=" * 60)

np.save(os.path.join(OUT_DIR, 'X_seq_train.npy'), X_seq_train)
np.save(os.path.join(OUT_DIR, 'X_cat_train.npy'), X_cat_train)
np.save(os.path.join(OUT_DIR, 'y_train.npy'),     y_train)
meta_train.to_csv(os.path.join(OUT_DIR, 'meta_train.csv'), index=False)

np.save(os.path.join(OUT_DIR, 'X_seq_test.npy'),  X_seq_test)
np.save(os.path.join(OUT_DIR, 'X_cat_test.npy'),  X_cat_test)
meta_test.to_csv(os.path.join(OUT_DIR, 'meta_test.csv'), index=False)

# Simpan config metadata untuk model
config = {
    'window_size': WINDOW_SIZE,
    'n_numerical_features': len(ALL_NUMERICAL),
    'n_pos': df_merged['pos_id'].nunique(),
    'n_landcover': df_merged['landcover_id'].nunique(),
    'feature_cols': ALL_NUMERICAL,
    'cat_cols': CATEGORICAL_COLS,
    'pos_classes': list(le_pos.classes_),
    'landcover_classes': list(le_lc.classes_),
}
with open(os.path.join(OUT_DIR, 'config.pkl'), 'wb') as f:
    pickle.dump(config, f)

print(f"\nSemua file tersimpan di: {OUT_DIR}")
print(f"  X_seq_train.npy  : {X_seq_train.nbytes / 1e6:.1f} MB")
print(f"  X_seq_test.npy   : {X_seq_test.nbytes / 1e6:.1f} MB")
print("\n" + "=" * 60)
print("PREPROCESSING SELESAI!")
print("=" * 60)
print(f"\nConfig model:")
print(f"  window_size           : {config['window_size']}")
print(f"  n_numerical_features  : {config['n_numerical_features']}")
print(f"  n_pos (embedding vocab): {config['n_pos']}")
print(f"  n_landcover (embed vocab): {config['n_landcover']}")
